# Hyperparameter Heston
## Lambda


In [1]:
# Standard library imports
import os
import time
from pathlib import Path

# Data manipulation and mathematics
import numpy as np
import pandas as pd
from scipy.stats import norm
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
from matplotlib import cm

# Deep learning framework (PyTorch)
import torch
import torch.nn as nn
import torch.optim as optim

# Sweep
import itertools


In [2]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_url = "https://github.com/egil10/fys5429.git"
    repo_dir = "/content/fys5429"

    if not os.path.exists(repo_dir):
        !git clone {repo_url} {repo_dir}
    else:
        !git -C {repo_dir} pull

    os.chdir(os.path.join(repo_dir, "code", "notebooks"))

print(f"Working directory: {os.getcwd()}")


Cloning into '/content/fys5429'...
remote: Enumerating objects: 792, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 792 (delta 81), reused 95 (delta 40), pack-reused 654 (from 1)
Receiving objects: 100% (792/792), 63.97 MiB | 42.37 MiB/s, done.
Resolving deltas: 100% (383/383), done.
Working directory: /content/fys5429/code/notebooks


### Colab setup


In [3]:
# Pathways
data_path = Path("..") / "data" / "generated" / "hs_collocation.parquet"

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    out_dir = Path("/content/drive/MyDrive/GITHUB-COLAB/fys5429/code/plots/eda")
else:
    out_dir = Path("..") / "plots" / "eda"

out_dir.mkdir(parents=True, exist_ok=True)
print(f"Plots will be saved to: {out_dir}")


Mounted at /content/drive
Plots will be saved to: /content/drive/MyDrive/GITHUB-COLAB/fys5429/code/plots/eda


In [4]:
# Importing HSPINN() Class
import sys
sys.path.insert(0, "../scripts")
from hspinn import HSPINN
from train_hs import train_pinn


### Global parameters


In [5]:
# Answer to the universe and everything
torch.manual_seed(42)

# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Adding torch backends
torch.backends.cudnn.benchmark = True

# Option Physics for Heston (Fixed baseline)
S_max = 300.0
T_max = 1.0
K = 100.0
r = 0.05
v_min = 0.01
v_max = 1.0
v0 = 0.04 

# Heston specific parameters (Fixed baseline)
kappa = 2.0    # mean-reversion speed
theta = 0.04   # long-run variance
xi = 0.3       # vol-of-vol
rho = -0.7     # correlation

# NN (Updated from Sweep 1 and 2 winners!)
HIDDEN_LAYERS = 3
NEURONS_PER_LAYER = 256
ACTIVATION = 'tanh'
LEARNING_RATE = 5e-3

# Epochs and sweeps
LAMBDA_PDE_VALUES = [10.0, 15.0, 20.0, 25.0, 30.0]
LAMBDA_IC_VALUES  = [5.0, 10.0, 15.0, 20.0]
LAMBDA_BC_FIXED   = 5.0

SWEEP_EPOCHS      = 5000


Using device: cuda


In [6]:
# Check if data exists
if not data_path.exists():
    raise FileNotFoundError(f"Collocation data not found at {data_path}. Please run generate_hs.ipynb first.")
else:
    df_all = pd.read_parquet(data_path)
    print(f"Loaded existing data from {data_path} (S_max={df_all['S'].max():.0f})")

# Extract tensors for Heston (S, v, tau)
def extract_tensors(df_subset):
    S_tensor = torch.tensor(df_subset['S'].values, dtype=torch.float32).view(-1, 1).to(device).requires_grad_(True)
    v_tensor = torch.tensor(df_subset['v'].values, dtype=torch.float32).view(-1, 1).to(device).requires_grad_(True)
    tau_tensor = torch.tensor(df_subset['tau'].values, dtype=torch.float32).view(-1, 1).to(device).requires_grad_(True)
    return S_tensor, v_tensor, tau_tensor

df_interior = df_all[df_all['point_type'] == 'interior']
S_in, v_in, tau_in = extract_tensors(df_interior)

df_ic = df_all[df_all['point_type'] == 'initial_condition']
S_ic, v_ic, tau_ic = extract_tensors(df_ic)

# Combine all boundary condition points (S_lower, S_upper, v_lower, v_upper)
df_bc = df_all[df_all['point_type'].str.startswith('boundary')]
S_bc, v_bc, tau_bc = extract_tensors(df_bc)

print(f"Interior points: {len(S_in)}")
print(f"IC points:       {len(S_ic)}")
print(f"BC points:       {len(S_bc)}")


Loaded existing data from ../data/generated/hs_collocation.parquet (S_max=300)
Interior points: 10000
IC points:       2000
BC points:       1000


In [7]:
sweep_results = []
start_time = time.time()
total_runs = len(LAMBDA_PDE_VALUES) * len(LAMBDA_IC_VALUES)

header = f"{'#':>3} | {'λ_PDE':>6} {'λ_IC':>6} | {'PDE Loss':>12} {'IC Loss':>12} {'BC Loss':>12} | {'Time':>6} {'ETA':>8}"
print(header)
print("─" * len(header))

for i, (lam_pde, lam_ic) in enumerate(itertools.product(LAMBDA_PDE_VALUES, LAMBDA_IC_VALUES)):

    run_start = time.time()
    result = train_pinn(S_in, v_in, tau_in, 
                        S_ic, v_ic, tau_ic, 
                        S_bc, v_bc, tau_bc,
                        r, K, kappa, theta, xi, rho, 
                        device,
                        lam_pde, lam_ic, LAMBDA_BC_FIXED, SWEEP_EPOCHS,
                        lr=LEARNING_RATE, hidden_layers=HIDDEN_LAYERS,
                        neurons=NEURONS_PER_LAYER, activation='tanh')

    result['lambda_pde'] = lam_pde
    result['lambda_ic'] = lam_ic
    result['lambda_bc'] = LAMBDA_BC_FIXED
    sweep_results.append(result)

    run_sec = time.time() - run_start
    total_elapsed = time.time() - start_time
    eta = (total_elapsed / (i + 1)) * (total_runs - i - 1)

    print(f"{i+1:>3} | {lam_pde:>6.1f} {lam_ic:>6.1f} | "
          f"{result['final_pde']:>12.6f} {result['final_ic']:>12.6f} {result['final_bc']:>12.6f} | "
          f"{run_sec:>5.0f}s {eta:>6.0f}s")

elapsed = time.time() - start_time
print("─" * len(header))
print(f"Sweep complete: {len(sweep_results)} runs in {int(elapsed//60)}m {int(elapsed%60):02d}s")


  # |  λ_PDE   λ_IC |     PDE Loss      IC Loss      BC Loss |   Time      ETA
──────────────────────────────────────────────────────────────────────────────


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:330.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


  1 |   10.0    5.0 |     6.149608    95.408875   181.299377 |   392s   7451s
  2 |   10.0   10.0 |     5.348345    29.035912   193.594131 |   397s   7099s
  3 |   10.0   15.0 |     7.385739    17.894756   221.656860 |   396s   6713s
  4 |   10.0   20.0 |     8.320403    13.756186   229.233612 |   396s   6321s


KeyboardInterrupt: 

In [ ]:
# Having the sweeps in a dataframe
df_sweep = pd.DataFrame([{
    'lambda_pde': r['lambda_pde'],
    'lambda_ic': r['lambda_ic'],
    'pde_loss': r['final_pde'],
    'ic_loss': r['final_ic'],
    'bc_loss': r['final_bc'],
    'total_loss': r['final_total'],
} for r in sweep_results])

# Sorting the values by PDE Loss
df_sweep = df_sweep.sort_values('pde_loss')

# Printing out to the console
print(df_sweep.to_string(index=False))


In [ ]:
pivot = df_sweep.pivot_table(index='lambda_pde', columns='lambda_ic', values='pde_loss')

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd_r', ax=ax)
ax.set_title(f'Heston PDE Loss Heatmap (BC Fixed at {LAMBDA_BC_FIXED})')

plt.tight_layout()
plt.savefig(out_dir / "hyper_hs_lambda_heatmap.pdf", bbox_inches="tight")
plt.show()


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.flush_and_unmount()
    print("Google Drive flushed and unmounted safely.")
